# Точный фильтр Парето (ht = 1) против гауссовской аппроксимации (ht = 10)

$\theta$ -- 4 состояния, генератор с равными внедиагональными интенсивностями
$\lambda_{ij} = 1/(3 \cdot 3600)$ ($i \neq j$); среднее время пребывания в
состоянии -- 3600 единиц времени. $Y = (Y_1, Y_2)$ -- двумерное, обе
координаты условно (при фиксированном $\theta$) независимы и равномерны на
непересекающихся по состояниям интервалах.

Наблюдение на шаге -- один непрерывный (вещественнозначный) канал

$$X = \mathrm{loc} + \mathrm{scale} \cdot \varepsilon, \qquad
\varepsilon = 1 + \frac{Z}{\sqrt{12}}, \qquad
Z = \frac{P - m(\alpha)}{s(\alpha)},$$

где $P \sim \mathrm{Pareto\,I}(x_m{=}1,\ \alpha)$ с плотностью
$f_P(p) = \alpha / p^{\alpha+1}$ при $p \ge 1$ (иначе 0),
$m(\alpha) = \alpha/(\alpha-1)$, $s(\alpha) = \sqrt{\alpha/(\alpha-2)}/(\alpha-1)$
-- среднее и СКО $P$. Масштаб $x_m$ при стандартизации сокращается, поэтому
семейство параметризуется ОДНИМ параметром формы $\alpha$. Здесь
$\mathrm{loc} = Y_1$, $\mathrm{scale} = Y_2$, $\alpha = 2.5$ (константа
`ALPHA` в конфигах).

$Z$ имеет среднее 0 и дисперсию 1, откуда $E\varepsilon = 1$,
$\mathrm{Var}\,\varepsilon = 1/12$, а значит $EX = Y_1 + Y_2$,
$\mathrm{Var}\,X = Y_2^2/12$ -- те же моменты, что использует гауссовская
аппроксимация. Это закон УЖЕ ДИСКРЕТИЗОВАННОГО наблюдения на шаге, а не
приращение непрерывного процесса, получаемое предельным переходом --
подробнее см. ниже.

## Постановка сравнения

Одна и та же траектория $(\theta_t, Y_t)$ и один и тот же поток наблюдений
(сгенерированный конфигом `pareto_obs` с $h_t = 1$). Три фильтра:

| Фильтр | конфиг | $h_t$ | правдоподобие приращения | наблюдения |
|---|---|---|---|---|
| точный | `pareto_obs` | 1 | точная плотность $Y_1 + Y_2\varepsilon$ (Парето) | исходные, шаг 1 |
| аппрокс., сумма блока | `pareto_obs_approx` | 10 | гауссовская по двум моментам суммы блока | те же, просуммированные блоками по 10 |
| аппрокс., среднее блока (ЦПТ) | `pareto_obs_clt` | 10 | гауссовская по двум моментам среднего блока | те же блоки, осреднённые |

У аппроксимации два независимых источника ошибки -- негауссовость
правдоподобия и огрубление шага. При $h_t = 10$ второе частично лечит первое
(сумма/среднее 10 наблюдений ближе к гауссовской по ЦПТ), и вопрос в том, что
перевешивает. Третий фильтр (среднее блока) -- не ЕЩЁ ОДНА точка сравнения: он
тождествен второму, см. раздел "Третий фильтр" ниже.

## Точная плотность наблюдения

С $j = s(\alpha)\sqrt{12}/\mathrm{scale}$ и $p = m(\alpha) + j(x - \mathrm{loc} - \mathrm{scale})$

$$f_X(x;\, \mathrm{loc}, \mathrm{scale}, \alpha) = \frac{\alpha}{p^{\alpha+1}}\, j,
\qquad p \ge 1$$

(иначе 0), носитель $x \ge \mathrm{loc} + \mathrm{scale}\cdot e_{\min}(\alpha)$,
$e_{\min}(\alpha) = 1 + (1 - m(\alpha))/(s(\alpha)\sqrt{12})$; при $\alpha = 2.5$
$e_{\min} \approx 0.871$. Реализовано в `core/densities.py` как
`pareto_obs_pdf` и используется каналом `PARETO` конфига `pareto_obs`.

Если внутри шага происходит скачок $\theta$, канал бывает в нескольких
состояниях $(\mathrm{loc}_r, \mathrm{scale}_r, \alpha_r)$ с временами
пребывания $u_r$, $\sum_r u_r = h_t$. Парето не безгранично делимо, поэтому
правдоподобие шага -- не накопление параметров, а СМЕСЬ плотностей по
временам пребывания:

$$f(x) = \sum_r \frac{u_r}{h_t}\, f_X(x;\, \mathrm{loc}_r, \mathrm{scale}_r, \alpha_r).$$

Это точное определение модели при любом $h_t$, ограничение $h_t = 1$ не
требуется.

## Почему агрегирование наблюдений корректно

Наблюдение -- процесс с приращениями: приращение на $(0, 10]$ есть сумма
приращений на $(0,1], \dots, (9,10]$. Поэтому наблюдения для фильтров с
$h_t = 10$ получаются из блоков по 10 из ОДНОЙ и той же выборки
`pareto_obs`, а не новой генерацией. Для гауссовского канала это ещё и
самосогласовано: сумма 10 независимых наблюдений имеет среднее
$10(Y_1 + Y_2)$ и дисперсию $10 Y_2^2/12$ -- ровно то, что даёт линейное
накопление параметров `params = ht * C[m, y]` при $h_t = 10$. Значит,
аппроксимационный фильтр при $h_t = 10$ -- это честная ЦПТ-аппроксимация
суммы 10 наблюдений, и она ОБОСНОВАННЕЕ, чем та же гауссовская
аппроксимация при $h_t = 1$.

## Третий фильтр: осреднение блока (ЦПТ) и его тождественность суммирующему

`pareto_obs_clt` получает вместо суммы блока его СРЕДНЕЕ

$$\bar X_j = \frac{1}{c}\sum_{i=1}^{c} X_{10j+i}, \qquad c = 10,$$

для которого по ЦПТ $E\bar X = Y_1+Y_2$, $\mathrm{Var}\,\bar X = Y_2^2/(12c)$.
Ядро фильтра всегда строит `params = ht * C`, поэтому, чтобы получить закон
среднего (а не суммы) при $h_t = c = 10$, скорости накопления надо поделить
на $c$:

$$\mathrm{drift} = \frac{Y_1+Y_2}{c}, \qquad \mathrm{var} = \frac{Y_2^2}{12c^2}.$$

**Это преобразование не меняет результат фильтрации.** Осреднение -- это
детерминированное аффинное преобразование наблюдения $x \to x/c$,
согласованно применённое и к параметрам канала. Для нормальной плотности
$p(x/c;\, \mu/c,\, \sigma^2/c^2) = c \cdot p(x;\, \mu, \sigma^2)$, то есть КАЖДОЕ
слагаемое ядра фильтра (по числу скачков $\theta$ внутри шага: $r=0,1,2$)
домножается на ОДИН И ТОТ ЖЕ множитель $c$, не зависящий ни от узла сетки, ни
от состояния $\theta$. Этот общий множитель сокращается при нормировке
$\psi$, поэтому апостериорное распределение -- и оценки $\theta$, $Y$ -- у
`pareto_obs_approx` (сумма блока) и `pareto_obs_clt` (среднее блока)
совпадают с точностью до ошибок округления с плавающей точкой. Третий
фильтр показывает эту тождественность, а не новый результат сравнения.

## Про `two_jumps`

При $T = 24 \cdot 3600$ и $h_t = 1$ точный фильтр делает 86400 шагов;
слагаемое с двумя скачками $\theta$ за шаг (`two_jumps=True`) при $N = 4$ и
сетке $20 \times 20 = 400$ узлов даёт порядка $10^9$--$10^{10}$ вызовов
`integrand2` НА ШАГ, то есть $10^{14}$--$10^{15}$ за весь прогон --
совершенно неподъёмно. При этом $h_t |\lambda| = 1/3600 \approx 2.8
\cdot 10^{-4}$, то есть $P(\text{2 скачка за шаг}) \sim (h_t|\lambda|)^2/2
\approx 4 \cdot 10^{-8}$ -- пренебрежимо мало, поэтому `two_jumps = False`
во всех трёх конфигах.

In [ ]:
import _bootstrap  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

from discretized_filter.config import set_config
from discretized_filter.core.smjp import sparse_mc
from discretized_filter.core.filter import Filter
from discretized_filter.utils.grids import to_discrete
from discretized_filter.visualization.plots import (
    plot_theta_background, theta_labels_default, theta_colors_default,
)

cfg_approx = set_config('pareto_obs_approx')
cfg_clt = set_config('pareto_obs_clt')
cfg = set_config('pareto_obs')      # точный, активен последним

print(f'точный  : N={cfg.N}, M={cfg.M}, K={cfg.K}, ht={cfg.ht}, '
      f'шагов={cfg.t_net_filtering.shape[0]}, узлов сетки={cfg.M_net.shape[1]}')
print(f'аппрокс.: N={cfg_approx.N}, M={cfg_approx.M}, K={cfg_approx.K}, '
      f'ht={cfg_approx.ht}, шагов={cfg_approx.t_net_filtering.shape[0]}, '
      f'узлов сетки={cfg_approx.M_net.shape[1]}')
print(f'ЦПТ-сред.: N={cfg_clt.N}, M={cfg_clt.M}, K={cfg_clt.K}, '
      f'ht={cfg_clt.ht}, шагов={cfg_clt.t_net_filtering.shape[0]}, '
      f'узлов сетки={cfg_clt.M_net.shape[1]}')

In [ ]:
theta, y, t = sparse_mc(cfg.p0, cfg.Lambda, cfg.lam, cfg.T, cfg.get_y, cfg.y_intervals)
obs = cfg.get_obs(cfg.t_net_filtering, theta, y, t)

# наблюдения для фильтров с ht=10 -- не новая генерация, а суммирование
# блоков по 10 из ОДНОЙ и той же выборки obs (наблюдение -- процесс с
# приращениями, см. markdown выше, 'Почему агрегирование корректно')
ratio = int(round(cfg_approx.ht / cfg.ht))          # 10
n_blocks = obs.shape[0] // ratio
obs_agg = obs[:n_blocks * ratio].reshape(n_blocks, ratio, cfg.K).sum(axis=1)
assert n_blocks == cfg_approx.t_net_filtering.shape[0] - 1, (
    f'{n_blocks} блоков наблюдений против '
    f'{cfg_approx.t_net_filtering.shape[0] - 1} шагов аппроксимационного '
    'фильтра -- сетки конфигов разъехались (T/seed/N/Lambda/y_intervals/num1 '
    'должны совпадать у pareto_obs и pareto_obs_approx)'
)

# то же самое блочное окно, только осреднённое (а не просуммированное) --
# наблюдения для третьего (ЦПТ-осредняющего) фильтра pareto_obs_clt
obs_mean = obs_agg / ratio

print(f'скачков theta: {len(t)}')
header = 'выборка            ht        n       mean        std      max|x|'
print(header)
for label, ht_, o in [
    ('точная', cfg.ht, obs),
    ('агрег. (сумма 10)', cfg_approx.ht, obs_agg),
    ('ЦПТ (среднее 10)', cfg_clt.ht, obs_mean),
]:
    print(f'{label:<18} {ht_:5.0f} {o.shape[0]:7d} {o.mean():9.3f} '
          f'{o.std():9.3f} {np.abs(o).max():9.3f}')

In [ ]:
def run(config, observations):
    f = Filter(
        config.pi_init, config.pi, config.M_net, config.C,
        config.N, config.Lambda, config.ht, config.delta, config.obs_density,
        n_points=config.n_points, two_jumps=config.two_jumps,
        filter_step=config.filter_step,
    )
    est = f.estimate()
    th, yy = [est[0]], [est[1]]
    for obs_ in observations:
        f.update(obs_)
        est = f.estimate()
        th.append(est[0])
        yy.append(est[1])
    return np.array(th), np.array(yy)


# ВНИМАНИЕ О СТОИМОСТИ: при T=24*3600, ht=1 точный фильтр делает 86400 шагов,
# и на каждом шаге ядро оценивает плотность порядка N^2 * n_grid^2 * n_points
# = 16 * 400^2 * 2 ~ 5e6 раз -- полный прогон занимает часы. Оба фильтра с
# ht=10 (сумма и ЦПТ-среднее) дешевле ровно в 10 раз по числу шагов и делают
# ОДИНАКОВОЕ число шагов друг с другом, поэтому третий прогон почти не
# добавляет стоимости ко второму. Для пробного прогона уменьшите T во ВСЕХ
# трёх конфигах (configs/pareto_obs.py, configs/pareto_obs_approx.py,
# configs/pareto_obs_clt.py, например до 3*3600) или num1, перегенерировав
# конфиги и траекторию заново.
runs = {
    'точный, ht=1': run(cfg, obs),
    'аппрокс., ht=10': run(cfg_approx, obs_agg),
    'ЦПТ-среднее, ht=10': run(cfg_clt, obs_mean),
}

In [ ]:
th_exact, yy_exact = runs['точный, ht=1']
th_approx, yy_approx = runs['аппрокс., ht=10']
th_clt, yy_clt = runs['ЦПТ-среднее, ht=10']

# общая сетка сравнения -- ht=10 (сетка аппроксимационных фильтров);
# оценки точного фильтра (своя сетка ht=1) прореживаем с шагом ratio
dtheta = to_discrete(
    np.vstack([np.int64(theta == i) for i in range(cfg.N)]).T, t, cfg.T, cfg_approx.ht
)
dY = to_discrete(y, t, cfg.T, cfg_approx.ht)
n = dtheta.shape[0]

th_exact_thin = th_exact[::ratio]
yy_exact_thin = yy_exact[::ratio]

header = 'фильтр                    RMSE theta   RMSE Y1   RMSE Y2'
print(header)
for label, th, yy in [
    ('точный (прореж., ht=10)', th_exact_thin, yy_exact_thin),
    ('аппрокс., ht=10', th_approx, yy_approx),
    ('ЦПТ-среднее, ht=10', th_clt, yy_clt),
]:
    rt = np.sqrt(((dtheta - th[:n]) ** 2).mean())
    ry1 = np.sqrt(((dY[:, 0] - yy[:n, 0]) ** 2).mean())
    ry2 = np.sqrt(((dY[:, 1] - yy[:n, 1]) ** 2).mean())
    print(f'{label:<24} {rt:11.4f} {ry1:9.4f} {ry2:9.4f}')

# справочно: RMSE точного фильтра на ЕГО СОБСТВЕННОЙ сетке ht=1 (без
# прореживания) -- показывает, чего лишается точный фильтр при огрублении
# сетки сравнения до ht=10
dtheta1 = to_discrete(
    np.vstack([np.int64(theta == i) for i in range(cfg.N)]).T, t, cfg.T, cfg.ht
)
dY1 = to_discrete(y, t, cfg.T, cfg.ht)
n1 = dtheta1.shape[0]
rt1 = np.sqrt(((dtheta1 - th_exact[:n1]) ** 2).mean())
ry1_1 = np.sqrt(((dY1[:, 0] - yy_exact[:n1, 0]) ** 2).mean())
ry2_1 = np.sqrt(((dY1[:, 1] - yy_exact[:n1, 1]) ** 2).mean())
label1 = 'точный (своя сетка ht=1)'
print(f'{label1:<24} {rt1:11.4f} {ry1_1:9.4f} {ry2_1:9.4f}'
      '   <- справочно, другая сетка сравнения')

In [ ]:
styles = {
    'точный, ht=1': dict(color='tab:red', lw=1.0),
    'аппрокс., ht=10': dict(color='tab:blue', lw=1.4),
    # пунктир: тождественен предыдущей кривой (см. markdown), иначе они
    # неразличимы на графике
    'ЦПТ-среднее, ht=10': dict(color='tab:green', lw=1.4, ls='--'),
}

fig, axes = plt.subplots(cfg.N, 1, figsize=(13, 9), layout='constrained', sharex=True)
for n_, ax in enumerate(axes):
    plot_theta_background(
        ax, theta, t, theta_labels_default, theta_colors_default, 1, alpha=0.15
    )
    # каждый фильтр -- по своей временной сетке (ht=1 и ht=10), без подгонки
    ax.plot(cfg.t_net_filtering, th_exact[:, n_],
            label='точный, ht=1', **styles['точный, ht=1'])
    ax.plot(cfg_approx.t_net_filtering, th_approx[:, n_],
            label='аппрокс., ht=10', **styles['аппрокс., ht=10'])
    ax.plot(cfg_clt.t_net_filtering, th_clt[:, n_],
            label='ЦПТ-среднее, ht=10', **styles['ЦПТ-среднее, ht=10'])
    ax.set(ylabel=rf'$\hat\theta^{n_ + 1}_t$', ylim=(-0.05, 1.05))
axes[-1].set(xlabel='$t$', xlim=(0, cfg.T))
axes[0].legend(ncol=3, fontsize=8, loc='upper right')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), layout='constrained', sharex=True)
labels_y = ['$Y^1_t$', '$Y^2_t$']
for j, ax in enumerate(axes):
    ax.step([0] + list(t), [y[0, j]] + list(y[:, j]),
            where='pre', color='k', lw=2.5, alpha=0.35, label=labels_y[j])
    ax.plot(cfg.t_net_filtering, yy_exact[:, j],
            label='точный, ht=1', **styles['точный, ht=1'])
    ax.plot(cfg_approx.t_net_filtering, yy_approx[:, j],
            label='аппрокс., ht=10', **styles['аппрокс., ht=10'])
    ax.plot(cfg_clt.t_net_filtering, yy_clt[:, j],
            label='ЦПТ-среднее, ht=10', **styles['ЦПТ-среднее, ht=10'])
    ax.set(ylabel=rf'$\hat Y^{j + 1}_t$')
axes[-1].set(xlabel='$t$', xlim=(0, cfg.T))
axes[0].legend(ncol=4, fontsize=9, loc='lower right')
plt.show()

In [ ]:
# итог: разность точный (прореженный до ht=10) минус аппроксимационный на
# общей сетке ht=10
d_theta = th_exact_thin[:n] - th_approx[:n]
d_Y = yy_exact_thin[:n] - yy_approx[:n]

print(f'макс |точный(проред.) - аппрокс.| по theta = {np.abs(d_theta).max():.4f}')
print(f'макс |точный(проред.) - аппрокс.| по Y     = {np.abs(d_Y).max():.4f}')

# мера неадекватности гауссовской аппроксимации на ЕДИНИЧНОМ шаге (ht=1):
# доля исходных наблюдений (ht=1), выходящих за 5 сигма гауссовского канала
# NORMAL (drift/var канала pareto_obs_approx -- те же моменты, что фильтр
# видит при ht=10, но здесь проверяется отклонение по-шаговое при ht=1)
drift_fn = cfg_approx.channels[0].drift
var_fn = cfg_approx.channels[0].var
drift_disc = drift_fn(0, dY1[1:], 0)[:, 0]
std_disc = np.sqrt(var_fn(0, dY1[1:], 0))[:, 0]
outliers = np.abs(obs[:, 0] - drift_disc) > 5 * std_disc
print(f'доля наблюдений (ht=1) вне 5 сигма гауссовского канала: '
      f'{outliers.mean():.3%}')